In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---


# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 0 &mdash; GPGN 404 Syllabus: Digital Signal Processing</h1></center>

Welcome to GPGN 404 Digital Signal Processing! Here are some key details that I would like to share about this course for our first class. The authoritative syllabus, including all dates and term-specific details, is on the course Canvas page.

<a name="0.1"></a>
## 0.1 Course Information

<b>Instructor:</b>
<ul>
<li>Professor Jeffrey Shragge, Geophysics Department</li>
<li>Office, phone, email and office hours: see the course Canvas page</li>
</ul>

<b>Teaching assistant:</b> name, contact details and office hours: see the course Canvas page.

<b>Class meeting days, times and location:</b> lectures (2 hours per week) and lab (3 hours per week) <span style="color:#b03a2e; font-size:0.9em">[re-verify from current Canvas syllabus]</span>.

<b>Instructional activity:</b> 2 hours lecture, 3 hours lab; 3 semester hours.

<b>Course designation:</b> GPGN major requirement.

<b>Prerequisites:</b> MATH 213, MATH 225, and MATH 348 or PHGN 311.

<b>Related courses:</b> builds on GPGN 328 Physics of the Earth 1; usually taken alongside GPGN 329 Physics of the Earth 2.

<b>Required technology:</b> a computer that can run Python 3 and JupyterLab with the course's <code>dsp2026</code> environment
(setup instructions are in the repository's <a href="https://github.com/jshragge/CSM_GP_DIGSIG_2026#readme">README</a>), and access to Canvas.

<a name="0.2"></a>
## 0.2 Course Materials

There is no required textbook: <b>these course notes are the textbook.</b> They are Jupyter notebooks that combine the lecture material
with reproducible Python examples that use NumPy, SciPy and Matplotlib. Download them by running the following in the folder where you
want them, and run `git pull` before each class for updates:

    git clone --depth 1 https://github.com/jshragge/CSM_GP_DIGSIG_2026.git

The README in the repository explains how to create the `dsp2026` Python environment the notebooks need. Lab handouts are posted on Canvas.

<b>References (optional):</b>

* *Discrete-Time Signal Processing*, 3rd edition, by A. V. Oppenheim and R. W. Schafer. A very comprehensive textbook on the subject;
  however, it may be somewhat abstract for some readers, since it is written from an electrical engineering perspective.
* *The Scientist and Engineer's Guide to Digital Signal Processing*, 2nd edition, by Steven W. Smith, available
  [free online](http://www.dspguide.com/pdfbook.htm). Aimed at an upper-undergraduate level and written with a balance of science and
  engineering in mind.
* *A Student's Guide to Python for Physical Modeling*, by Jesse M. Kinder and Philip Nelson. A helpful guide for those starting down
  the path of Python programming. If you are at that point, I strongly recommend picking it up.

<a name="0.3"></a>
## 0.3 Course Description

The fundamentals of 1D and 2D digital signal processing as applied to geophysical investigations are studied. Students explore the mathematical background and practical consequences of the &hellip; <span style="color:#b03a2e; font-size:0.9em">[re-verify from current Canvas syllabus]</span>

The topics, in the order of the course modules, are:

* Complex Numbers
* Fourier Series
* 1D Fourier Transforms
* 2D Fourier Transforms
* LTI Systems, Convolution and Correlation
* Signals as Vectors, Systems as Matrices: the SVD
* Discrete-Time Signals
* Digital Sampling and Reconstruction
* Discrete Fourier Transform
* Windows and Spectrograms
* Wavelet Transforms
* Z-Transform
* Practical Filtering

Emphasis is placed on applying the knowledge gained in lecture to exploring practical (geophysical) signal processing issues. This is done through homework and in-class practicum assignments requiring the programming and testing of algorithms discussed in lecture.

The Bulletin text above is from the 2023 syllabus; its first sentence is incomplete there <span style="color:#b03a2e; font-size:0.9em">[re-verify from current Canvas syllabus]</span>.

<a name="0.4"></a>
## 0.4 Student Learning Outcomes

**Conceptual Skills:** This senior-level undergraduate course builds on analytical skills learned in previous years by applying those skills to the analysis of 1D and 2D digital signals with an emphasis on *geophysical* applications. Major objectives of the course include:

1. Learning and exploiting similarities between concepts learned in calculus, differential equations, and elsewhere, and how they appear in the context of *geophysical* digital signal analysis. (ABET 1,6)
2. Understanding how to use discrete Fourier transforms in the analysis and processing of digital signals. (ABET 1,6)
3. Learning how to digitally sample continuous analog signals, reconstruct continuous signals from sampled ones, and the conditions under which this reconstruction is feasible. (ABET 1,6)
4. Learning the pitfalls and tradeoffs in the design and application of common digital filters. (ABET 1,6)

**Practical Skills:** To help meet these objectives students will:

1. Independently design, develop, validate and apply computer programs to solve *geophysical* digital signal analysis and processing tasks using the Python language and its associated tool kits (e.g., NumPy, SciPy and Matplotlib). (ABET 1,6,7)
2. Gain experience in choosing and applying 1D/2D filters to achieve specific filtering tasks through a range of numerical exercises through independent project work and communicating your results through a technical writing exercise. (ABET 1,3,6)

<a name="0.5"></a>
## 0.5 Delivery and Pedagogy

The material is presented through two one-hour lectures per week and one lab session per week <span style="color:#b03a2e; font-size:0.9em">[re-verify from current Canvas syllabus]</span> and through self-study.  Students will be provided with the material (**Jupyter Notebook**) before classes and are expected to read through and reproduce worked examples before class.  These notebooks use short **Python** examples to illustrate the concepts introduced in class. Class time is focused on clarification and going through worked examples as a whole class and in small groups.

<b>How the notes are organized.</b> Each module notebook opens with an introduction and an outline. Blue boxes hold notes, proofs and
<i>Synthesis</i> boxes that connect the module to earlier modules and to your mathematics and physics courses; green <i>Distraction</i> boxes
are optional enrichment; yellow boxes flag common misconceptions and hold the three <i>Microquestions</i> at the end of each module, whose
worked answers are hidden in collapsible blocks. The Python code is hidden by default (use the <b>Show code</b> button). Animations play
once the notebook is trusted: run the animation's cell (or the whole notebook) once.

<a name="0.6"></a>
## 0.6 Course Assessment

<b>What I don't like about <i>traditional</i> grading:</b>

<ul>
<li>Humans learn through feedback loops; however, traditional grading does not provide an effective feedback pathway.</li>
<li>Student work does not typically receive helpful feedback.</li>
<li>It penalizes those with "farther back starting lines" and does not reward hard work in "catching up".</li>
<li>It is demotivating when students "tank" the first major assessment.</li>
<li>It can promote unhealthy relationships between students and faculty, and between students.</li>
</ul>

<b>Specifications-based grading</b> is an alternative grading practice that aims to present students with clear guidelines and assessment
rubrics for successful work submissions and, when falling short, to provide effective feedback for how to prepare the work for resubmission
to meet the established criteria (Clark and Talbert, 2023). GPGN 404 uses this approach, with assessment based on <b>3</b> categories:

<ul>
<li><b>Category 1: Quizzes.</b> Purpose: to demonstrate an understanding and use of the key principles covered in the course notes.</li>
<li><b>Category 2: Labs.</b> Purpose: to design, develop, validate and apply Python programs that solve geophysical digital signal
analysis and processing tasks.</li>
<li><b>Category 3: Project.</b> Purpose: to choose and apply 1D/2D filters and processing steps to a real data set (processing and imaging
multi-offset GPR data) and to communicate the results in a technical report.</li>
</ul>

<a name="0.6.1"></a>
### 0.6.1 Quizzes

<ul>
<li>Quizzes (Category 1) are taken <b>online through Canvas and are graded automatically</b>; you see your result on submission.</li>
<li>A quiz earns a "Pass" (and credit) for a score of at least 90% <span style="color:#b03a2e; font-size:0.9em">[re-verify from current Canvas syllabus]</span>; scores below the threshold do not receive credit.</li>
<li>You may retake a quiz to attain a passing mark until the quiz closes on Canvas.</li>
<li><b>There is no penalty for retaking a quiz; full credit is possible.</b></li>
<li>Quizzes are individual work, and <b>generative AI is not allowed on quizzes</b> (Section 0.9).</li>
</ul>

<a name="0.6.2"></a>
### 0.6.2 Labs and Project

<ul>
<li>Submitted labs and the project (Categories 2 and 3) are assessed as one of the following "grades":
<ol>
<li><b>Exceeds Expectations (EE)</b>: a high-quality submission that clearly meets all stated specifications with minimal-to-no need of
improvement. Receives credit for a "Pass" grade.</li>
<li><b>Meets Expectations (ME)</b>: the submitted work meets all stated specifications, though it could be improved as per the provided
feedback. Receives credit for a "Pass" grade.</li>
<li><b>Near Miss (NM)</b>: does not meet all stated specifications and does not receive credit for that submission. See the resubmission
policy below.</li>
<li><b>Incomplete (INC)</b>: meets few-to-no specifications and needs considerable improvement to be acceptable. Does not receive credit
for that submission. See the resubmission policy below.</li>
<li><b>Not Submitted (NS)</b>: not submitted. Cannot be submitted after the final posted date.</li>
</ol></li>
<li><b>Exceeds Expectations</b> grades can raise the final mark by a fractional letter grade (e.g., from B to B+ or A&minus; to A); see
the grading table below.</li>
</ul>

<a name="0.6.3"></a>
### 0.6.3 Late Submission Policy

<ul>
<li>Requests for due-date extensions should be sent <b>by email 24 hours before the scheduled due date</b> and will be considered case by
case.</li>
<li>Unless an extension is granted in advance, work not submitted by the due date receives an <b>Incomplete</b> grade.</li>
</ul>

<a name="0.6.4"></a>
### 0.6.4 Resubmission Policy

<ul>
<li>You will receive feedback on each lab and on the project that you can use to revise your work.</li>
<li>Resubmissions are accepted within 5 business days of receiving feedback <span style="color:#b03a2e; font-size:0.9em">[re-verify from current Canvas syllabus]</span>.</li>
<li>A <b>Meets Expectations</b>, <b>Near Miss</b> or <b>Incomplete</b> submission may be resubmitted, using the instructor and teaching
assistant feedback, for an improved grade.</li>
<li>Resubmissions must include a short summary of how the work was modified in response to the feedback.</li>
<li>Work may be <b>resubmitted up to two times</b>; only one resubmission of an <b>Incomplete</b> grade is permitted for the course
<span style="color:#b03a2e; font-size:0.9em">[re-verify from current Canvas syllabus]</span>.</li>
<li>A <b>Not Submitted</b> lab cannot be submitted after the posted final date (unless agreed with the instructor and/or TA).</li>
<li><b>There is no penalty for (acceptable) resubmission; full credit is possible.</b></li>
</ul>

<a name="0.6.5"></a>
### 0.6.5 Grading Policy

The letter grade is set by the number of "pass" assessments in each category. The thresholds for this term are given in the grading table
in the Canvas syllabus <span style="color:#b03a2e; font-size:0.9em">[re-verify from current Canvas syllabus]</span>; they will be reproduced in Table 0.1 below once set.

In [ ]:
# --- Grade-threshold table: number of "pass" assessments required for each letter grade (specifications grading). ---
# The thresholds are set each term. Fill GRADING from the current Canvas syllabus; until then the table is not shown.
GRADING = None
# Example of the expected form (POTE1/POTE2 layout; values deliberately left blank):
# GRADING = {'Grade': ['A', 'A-', 'B+', 'B', 'C', 'D', 'F'],
#            'Quizzes passed (of N)': [...], 'Labs passed (of N) (# EE)': [...], 'Project': [...]}
if GRADING is None:
    print('Grade thresholds for this term: see the grading table in the Canvas syllabus.')
else:
    style_table(pd.DataFrame(GRADING))

<b>Table 0.1.</b> Number of "pass" assessments in each category (quizzes, labs, project) required for each letter grade. Shown once the term's thresholds are filled in; until then, see Canvas <span style="color:#b03a2e; font-size:0.9em">[re-verify from current Canvas syllabus]</span>. <i>(Python-generated.)</i>

<a name="0.7"></a>
## 0.7 Course Schedule

The <i>tentative</i> schedule in Table 0.2 follows the order of the course modules. Weekly dates, holidays and due dates are on Canvas <span style="color:#b03a2e; font-size:0.9em">[re-verify from current Canvas syllabus]</span>.

In [ ]:
# --- Tentative schedule table. Edit SCHEDULE, not the table. Dates are term-specific: see Canvas. ---
import pandas as pd
SCHEDULE = [(1, '01 Introduction; 02 Signals, Systems and Processing; 03 Complex Numbers', 'Lab 1 - Python Refresher and Complex Numbers', '-'), (2, '03 Complex Numbers; 04 Fourier Series', 'Lab 2 - Fourier Series', 'Quiz 1 - Complex Numbers'), (3, '04 Fourier Series; 05 1D Fourier Transforms', 'Lab 3 - 1D Fourier Transforms', 'Quiz 2 - Fourier Series'), (4, '05 1D Fourier Transforms; 06 2D Fourier Transforms', 'Lab 4 - 2D Fourier Transforms', 'Quiz 3 - 1D Fourier Transforms'), (5, '07 LTI Systems, Convolution and Correlation', 'Lab 5 - 1D/2D Convolution, Correlation and Deconvolution', 'Quiz 4 - 2D Fourier Transforms'), (6, '07 LTI Systems, Convolution and Correlation', 'Lab 6 - 2D Convolution, Linearity and Noise', 'Quiz 5 - 1D Convolution'), (7, '08 Signals as Vectors, Systems as Matrices: the SVD', 'Lab 7 - Singular Value Decomposition (SVD) and Applications', 'Quiz 6 - Correlation and Convolution'), (8, '08 Signals as Vectors, Systems as Matrices: the SVD; 09 Discrete-Time Signals', 'Lab 8 - Matrix Operators, Invertibility and Adjoints', 'Quiz 7 - SVD and Applications'), (9, '09 Discrete-Time Signals; 10 Digital Sampling and Reconstruction', 'Lab 9 - Sampling, Reconstruction and Random Sampling', 'Quiz 8 - Discrete-Time Signals'), (10, '11 Discrete Fourier Transform', 'Lab 10 - Discrete Fourier Transform (DFT)', 'Quiz 9 - Digital Sampling and Reconstruction'), (11, '-', 'Spring Break (no classes)', '-'), (12, '12 Windows and Spectrograms; 13 Wavelet Transforms', 'Lab 11 - Spectrograms and Time-Frequency Processing', 'Quiz 10 - Discrete Fourier Transform'), (13, '13 Wavelet Transforms; 14 Z-Transform', 'Lab 12 - Wavelet Transform', 'Quiz 11 - Windows and Spectrograms'), (14, '14 Z-Transform; 15 Practical Filtering', 'Lab 13 - Z-Transforms and FIR Filtering', 'Quiz 12 - Wavelet Transforms'), (15, '-', 'Project - Processing and imaging multi-offset GPR', 'Quiz 13 - Z-Transform and FIR/IIR Filtering'), (16, '-', 'Project - Processing and imaging multi-offset GPR', '-'), (17, '-', 'Project - Processing and imaging multi-offset GPR', '-')]

def style_table(df):
    styles = [{'selector': 'th', 'props': [('background-color', DSP_COLORS['oceanic_crust']), ('color', 'white'),
                                          ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '6px 10px'),
                                          ('border', '1px solid #ccc')]},
              {'selector': 'td', 'props': [('text-align', 'left'), ('padding', '5px 10px'), ('border', '1px solid #ccc')]},
              {'selector': '', 'props': [('width', '100%'), ('border-collapse', 'collapse')]}]
    return df.style.hide(axis='index').format(escape='html').set_table_styles(styles)

schedule_df = pd.DataFrame(SCHEDULE, columns=['Week', 'Modules', 'Lab / Project', 'Quiz'])
style_table(schedule_df)

<b>Table 0.2.</b> Tentative course schedule for GPGN 404: modules, labs and quizzes by week of the semester. Module titles match the course repository; lab titles are from the 2023 offering, and dates are on Canvas <span style="color:#b03a2e; font-size:0.9em">[re-verify from current Canvas syllabus]</span>. <i>(Python-generated.)</i>

<a name="0.8"></a>
## 0.8 How to Get Help

You should not hesitate to seek my help. You can do this in the following ways:
<ol>
<li>Ask questions in class. Never hesitate to ask, even if I have explained the material before. You can safely assume that if something is
not clear to you, several classmates have the same question.</li>
<li>Drop by my office, preferably during office hours.</li>
<li>Drop by the TA's office, preferably during office hours.</li>
<li>I will not provide help on labs or quizzes by email. If anything is not clear, or if you need help, ask in a timely way using any of
the three options above.</li>
</ol>

<a name="0.9"></a>
## 0.9 Generative Artificial Intelligence

At Mines, we are navigating the impacts of generative AI (genAI) together. My course policy for these tools follows from the skills you are
here to develop in GPGN 404. GenAI tools, including but not limited to ChatGPT, Claude, Gemini and Copilot, are:

<div class="alert alert-info">
<b>Allowed, with disclosure, on labs and the project (Categories 2 and 3).</b>
<ul>
<li><b>Disclose every use.</b> End each lab and the project report with a short <i>AI-use statement</i>: the tool (and version), what you
used it for (for example, debugging a plotting error, explaining a filter design, or editing a paragraph), and which parts of the submission
it affected. If you used none, write "No genAI used."</li>
<li><b>You own the result.</b> You are responsible for the correctness of everything you submit. Code must run, and you must be able to
explain any line of it, and any figure it makes, if asked.</li>
<li><b>Verify, don't trust.</b> GenAI writes signal-processing code that runs but is wrong, for example a frequency axis computed without the
sample interval, or a filter applied twice. Check your results against the tests used in the notes (a known sinusoid, an impulse,
Parseval's theorem).</li>
<li><b>Data are data.</b> Never present AI-generated or AI-altered data, figures or results as measurements or as your own processing output.</li>
<li><b>Cite it</b> using the formats in <a href="https://libguides.mines.edu/citing/AI">Citing AI</a> (Arthur Lakes Library).</li>
</ul>
</div>

<div class="alert alert-warning">
<b>Not allowed on quizzes (Category 1).</b> Quizzes are online, automatically graded checks of your own understanding. Using a genAI tool
(or any other person) to answer a quiz question is Academic Misconduct. Because quizzes can be retaken without penalty (Section 0.6.1), an
honest attempt always costs you less than a shortcut.
</div>

<b>Why this policy, in this course.</b> Professional signal processing is now done alongside AI coding assistants, and learning to use them
critically is part of the job. What this course still asks of you is the judgment the tools lack: choosing a sampling rate, a window or a
filter, and predicting the artifacts. Undisclosed use, use on a quiz, or use that misrepresents AI output as your own work is Academic
Misconduct under the <a href="https://www.mines.edu/policy-library/academic-integrity-policy/">Mines Academic Integrity Policy</a>. For more
information, see the <a href="https://academicaffairs.mines.edu/genai/"><i>Guidance for Mines Faculty and Students on Generative AI Use</i></a>.

<b>My commitment as your instructor.</b> My primary goal is to support your learning and growth throughout this course. I may use genAI tools
to help with that goal, for example to brainstorm examples and activities, or to make the course materials more accessible. When I use AI, I
commit to do so ethically, transparently and with human oversight. I will <b>not</b> use AI to replace my role as your instructor, to offload
my responsibility to assess your work carefully, or to input your identifying information into any unlicensed AI tool without your explicit
consent.

<a name="0.10"></a>
## 0.10 Campus Policies and Resources

The GPGN 404 policies and campus resources are reproduced below so this notebook is <b>self-contained</b>; the authoritative versions live in
the Syllabus on the course Canvas page and in the <a href="https://www.mines.edu/policy-library/">Mines Policy Library</a>. This section
follows the <b>Fall 2026 Mines syllabus template</b> <span style="color:#b03a2e; font-size:0.9em">[re-verify from current Canvas syllabus]</span>. <b>Click any item to jump to it.</b>

<a id="toc-010"></a>
<ol>
  <li><a href="#p00-01">Absences</a></li>
  <li><a href="#p00-02"><b>Sexual Misconduct, Discrimination, and Retaliation</b></a> <b>*</b></li>
  <li><a href="#p00-03">Academic Integrity</a></li>
  <li><a href="#p00-04"><b>Generative Artificial Intelligence</b></a> <b>*</b></li>
  <li><a href="#p00-05">Copyrighted Materials</a></li>
  <li><a href="#p00-06">Grading Policy</a></li>
  <li><a href="#p00-07">Coursework Return Policy</a></li>
  <li><a href="#p00-08">Course Issues and Concerns</a></li>
  <li><a href="#p00-09"><b>Disability Support Services</b></a> <b>*</b></li>
  <li><a href="#p00-10">Digital Accessibility</a></li>
  <li><a href="#p00-11">Student Outreach &amp; Support (SOS) Resources</a></li>
  <li><a href="#p00-12">Center for Academic Services and Advising (CASA)</a></li>
  <li><a href="#p00-13">The Writing Center</a></li>
  <li><a href="#p00-14">Tutoring Resources</a></li>
  <li><a href="#p00-15">Expectations for Participation</a></li>
</ol>

<b>Note *: Indicates action / discussion item.</b>

<hr>

<a id="p00-01"></a>
<h4>1. Absences <span style="font-size:0.82em">&nbsp;[<a href="#toc-010">back to list &uarr;</a>]</span></h4>
<p>Mines students are expected to fulfill their academic requirements through attendance and/or participation. Class attendance is required
of all students unless the student has an excused absence granted by the school or the student's professor. It is the student's
responsibility to request an excused absence with the professor or through the established protocol. Excused absences may be issued for
unforeseen illness or anticipated and/or authorized medical or personal matters including religious observance, military commitments, and
other qualifying reasons (upon review, exclusions apply). Opportunistic or habitual abuse of the excused absence policy violates the Mines
Code of Conduct.</p>
<p>To review the Excused Absence Policy and/or to request an excused absence, please visit
<a href="https://www.mines.edu/student-life/student-absences/">mines.edu/student-life/student-absences</a>.</p>
<p><b>For GPGN 404 specifically:</b> if an excused absence affects a quiz or lab deadline, contact me as early as you can so we can agree on
a revised date. The specifications-grading structure of this course (quiz retakes and lab resubmissions, Section 0.6) is designed to absorb
the occasional disruption without penalty.</p>

<a id="p00-02"></a>
<h4>2. Sexual Misconduct, Discrimination, and Retaliation <b>*</b> <span style="font-size:0.82em">&nbsp;[<a href="#toc-010">back to list &uarr;</a>]</span></h4>
<p>Mines prohibits discrimination and harassment on the basis of age, ancestry, creed, marital status, race, color, ethnicity, religion,
national origin, sex (including stalking, dating and domestic violence, non-consensual sexual contact or penetration, sexual exploitation, and
sexual harassment), pregnant or parenting status, gender, gender identity, gender expression, disability, sexual orientation, genetic
information, veteran status, or military service within any of its education programs or activities (including admissions and employment),
or sponsored conferences and events. This prohibition applies to all students, employees, contractors, visitors (including conference
attendees), and volunteers.</p>
<p>Please note that faculty and instructors are <b>mandatory reporters</b> and are required to report disclosures to the Title IX
Coordinator.</p>
<p>Anyone in our campus community can report or file a complaint of discrimination or sexual misconduct. Please reach out to
<a href="mailto:OIE@Mines.edu">OIE@Mines.edu</a> or file through the
<a href="https://cm.maxient.com/reportingform.php?COSchoolofMines&amp;layout_id=31">Discriminatory Harassment and Retaliation Reporting
Form</a>. Confidential resources are available at Mines and listed on the
<a href="https://oie.mines.edu/resources/">Office for Institutional Equity resources page</a>.</p>
<p><b>Preferred First Name.</b> Mines recognizes members of the campus community may prefer to use a first name other than their legal name
to identify themselves. Many services on campus, like Canvas, utilize and display preferred first names. Additional information can be found
on the Office for Institutional Equity <a href="https://oie.mines.edu/preferred-first-name/">Preferred First Name page</a>, or you can reach
out directly to the Title IX Coordinator at <a href="mailto:OIE@Mines.edu">OIE@Mines.edu</a>. <i>In this course:</i> course rosters reach me
with your legal name. Please tell me early in the semester if you would like me to address you by a different name or by particular pronouns,
and I will update my records.</p>
<p><b>Pregnant and Parenting.</b> If you are a pregnant or parenting student, you have the right to reasonable, non-disability related
modifications. Please contact the Title IX Coordinator in the Office for Institutional Equity at <a href="mailto:oie@mines.edu">oie@mines.edu</a>
for more information and to request support and/or modifications.</p>

<a id="p00-03"></a>
<h4>3. Academic Integrity <span style="font-size:0.82em">&nbsp;[<a href="#toc-010">back to list &uarr;</a>]</span></h4>
<p>Colorado School of Mines affirms the principle that all individuals associated with the Mines academic community have a responsibility for
establishing, maintaining, and fostering an understanding and appreciation for academic integrity. Please read the
<a href="https://www.mines.edu/policy-library/academic-integrity-policy/">full academic misconduct/integrity policy</a> for full definitions of
academic misconduct. Additionally, please use the
<a href="https://www.mines.edu/student-life/community-standards/academic-integrity/">resources provided by the Department of Community
Standards</a> for guidance on the procedures of the policy.</p>
<p><b>Course-specific expectations for GPGN 404.</b></p>
<ul>
  <li><b>Quizzes</b> are individual work, without collaboration and without genAI. Retakes are unlimited and unpenalized, so there is never an
      incentive to seek an unauthorized shortcut.</li>
  <li><b>Labs and the project</b>: discussing concepts and approaches with classmates is encouraged and is part of how engineers work. The code,
      figures, analyses and writing you submit must be produced by you (with any genAI use disclosed, Section 0.9). Where a classmate's idea
      shaped your solution, say so in a line of acknowledgement; this is normal professional practice and is never penalized.</li>
  <li><b>Sources</b> (the notes, papers, web pages, code you did not write) must be cited, including when you have paraphrased rather than
      quoted.</li>
</ul>

<a id="p00-04"></a>
<h4>4. Generative Artificial Intelligence <b>*</b> <span style="font-size:0.82em">&nbsp;[<a href="#toc-010">back to list &uarr;</a>]</span></h4>
<p>The GPGN 404 genAI policy is given in full in <b>Section 0.9</b> above: genAI is <b>allowed with disclosure</b> on labs and the project,
and <b>not allowed on quizzes</b>. Representing AI-generated content as your own original work without disclosure and citation, or using it
where it is not allowed, is Academic Misconduct under the Mines Academic Integrity Policy.</p>

<a id="p00-05"></a>
<h4>5. Copyrighted Materials <span style="font-size:0.82em">&nbsp;[<a href="#toc-010">back to list &uarr;</a>]</span></h4>
<p>Readings, lectures, media, data, and software used in this course may be protected by copyright law. You should not post, share, copy,
distribute, upload, or publicly display these materials outside of class without permission from the copyright owner. Unauthorized use may
result in <a href="https://www.mines.edu/policy-library/copyright-infringement-and-illegal-file-sharing/">disciplinary action</a> and legal
liability. This applies to lab handouts and data sets distributed through Canvas.</p>

<a id="p00-06"></a>
<h4>6. Grading Policy <span style="font-size:0.82em">&nbsp;[<a href="#toc-010">back to list &uarr;</a>]</span></h4>
<p>The full grading policy for GPGN 404 is given in <b>Section 0.6 Course Assessment</b> above: three assessment categories, pass/no-pass on
each item against published specifications, with the letter grade determined by the number of passed assessments in each category. Quizzes
may be retaken and labs resubmitted, without penalty, under the conditions in Sections 0.6.1 and 0.6.4.</p>

<a id="p00-07"></a>
<h4>7. Coursework Return Policy <span style="font-size:0.82em">&nbsp;[<a href="#toc-010">back to list &uarr;</a>]</span></h4>
<p>Canvas quizzes are graded automatically, and you see your result on submission. Labs and the project are returned with a
specifications-based grade (EE / ME / NM / INC) and written feedback <b>within one week</b> of the submission deadline <span style="color:#b03a2e; font-size:0.9em">[re-verify from current Canvas syllabus]</span>. Because the
resubmission window starts when you receive feedback (Section 0.6.4), prompt return of your work is part of how the feedback loop in this course
is designed to function.</p>

<a id="p00-08"></a>
<h4>8. Course Issues and Concerns <span style="font-size:0.82em">&nbsp;[<a href="#toc-010">back to list &uarr;</a>]</span></h4>
<p>As part of good professional practice, students are encouraged to speak with the faculty directly to raise issues and concerns with regard
to the course, professionally and in compliance with the student code of conduct. Students can also reach out to the head of the department
through which the course is offered, Paul Sava <span style="color:#b03a2e; font-size:0.9em">[re-verify from current Canvas syllabus]</span>. The department head can investigate and work with the faculty member to resolve
course-related concerns. Students' final point of contact is the college dean, Terri Hogue <span style="color:#b03a2e; font-size:0.9em">[re-verify from current Canvas syllabus]</span>, who can make any final decisions.</p>

<a id="p00-09"></a>
<h4>9. Disability Support Services <b>*</b> <span style="font-size:0.82em">&nbsp;[<a href="#toc-010">back to list &uarr;</a>]</span></h4>
<p>Disability Support Services (DSS) works collaboratively with students, faculty, and staff to minimize barriers and support an accessible
campus community. When barriers to access occur, Disability Support Services works one-on-one with students to determine accommodations and
facilitate access to programs and services. If you have been approved for accommodations through Disability Support Services, please contact
me to confirm receipt of your accommodation letter and to discuss the implementation of accommodations in this course. Please visit
<a href="https://www.mines.edu/disability-support-services/">mines.edu/disability-support-services</a> for more information or to request
accommodations.</p>
<p>If you anticipate or experience any barrier to learning in this course, whether or not you are registered with DSS, you are welcome to
discuss it with me directly.</p>

<a id="p00-10"></a>
<h4>10. Digital Accessibility <span style="font-size:0.82em">&nbsp;[<a href="#toc-010">back to list &uarr;</a>]</span></h4>
<p>The Colorado School of Mines is committed to supporting an accessible digital environment for all members of our community, including
students with disabilities. If you have an accessibility concern with Canvas or any digital materials or software used in this course, please
contact me or request support from Information Technology. Please visit <a href="https://www.mines.edu/accessibility/">mines.edu/accessibility</a>
for more information.</p>
<p>This course is delivered largely through Jupyter notebooks with Python-generated figures, animations and audio. If any figure, animation,
color scheme, or sound example is difficult for you to read, hear or use, please tell me: the source code is open, and I can adjust color maps
and font sizes or provide an alternative.</p>

<a id="p00-11"></a>
<h4>11. Student Outreach &amp; Support (SOS) Resources <span style="font-size:0.82em">&nbsp;[<a href="#toc-010">back to list &uarr;</a>]</span></h4>
<p>If you feel overwhelmed, anxious, depressed, or concerned about your overall well-being, you can connect with an SOS Case Manager who can
help navigate the resources available to support you. If you need assistance, please ask for help from a trusted faculty or staff member,
fellow student, or submit a <a href="https://cm.maxient.com/reportingform.php?COSchoolofMines&amp;layout_id=20">referral</a> for yourself. As a
community, we can help one another get through difficult times. If you are concerned for another student, offer assistance and/or ask for help
on their behalf. Students seeking resources for themselves or others should visit
<a href="https://student-life.mines.edu/student-outreach-and-support/">student-life.mines.edu/student-outreach-and-support</a>.</p>
<p>Student Outreach and Support can help connect you with a variety of resources; some of those might include:</p>
<ul>
  <li><b>Counseling Center</b>: <a href="https://counseling.mines.edu/">counseling.mines.edu</a>. Located on the 2nd floor of the Wellness
      Center, 1770 Elm St.</li>
  <li><b>Health Center</b>: <a href="https://studenthealth.mines.edu/">studenthealth.mines.edu</a>. Located on the 1st floor of the Wellness
      Center.</li>
  <li><b>SHAPE Office</b> (sexual and interpersonal violence support): <a href="https://shape.mines.edu/">shape.mines.edu</a>, or
      <a href="mailto:confidentialresource@mines.edu?subject=Confidential%20support">email</a> to make an appointment. Located on the 2nd floor
      of the Wellness Center.</li>
  <li><b>988 Colorado: Colorado Mental Health Line</b>: for support with any mental health and/or substance use concerns you can call or text
      <b>988</b>, visit a walk-in center, or chat with someone via <a href="https://www.988colorado.com/en">988colorado.com</a>.</li>
  <li><b>Food and/or housing insecurity</b>: any student who has difficulty securing food or housing, and believes this may affect their
      performance in the course, is urged to contact Student Outreach and Support. Please also notify me if you are comfortable doing so.</li>
</ul>
<p><b>In an emergency, call 911</b>, and they will dispatch a Mines or Golden PD officer to assist.</p>

<a id="p00-12"></a>
<h4>12. Center for Academic Services and Advising (CASA) <span style="font-size:0.82em">&nbsp;[<a href="#toc-010">back to list &uarr;</a>]</span></h4>
<p>CASA provides a variety of academic support services for students during their time at Mines. Please visit
<a href="https://www.mines.edu/casa/academic-support/">mines.edu/casa/academic-support</a> for a complete list of current academic support
services. For questions please email <a href="mailto:casa@mines.edu">casa@mines.edu</a>.</p>

<a id="p00-13"></a>
<h4>13. The Writing Center <span style="font-size:0.82em">&nbsp;[<a href="#toc-010">back to list &uarr;</a>]</span></h4>
<p>The Writing Center is a free academic support service for undergraduate and graduate students. Professional consultants and peer tutors
provide support with all forms of communication, including technical and scientific reports, academic essays, and oral presentations. Students
can make an online or in-person appointment at any stage of their project, from brainstorming to final revisions. This is a good resource for
the project report. To learn more and to make an appointment, please visit <a href="https://writing.mines.edu/">writing.mines.edu</a>.</p>

<a id="p00-14"></a>
<h4>14. Tutoring Resources <span style="font-size:0.82em">&nbsp;[<a href="#toc-010">back to list &uarr;</a>]</span></h4>
<p>The Mines Tutoring page will direct you to tutoring locations, hours, and offices most appropriate for your needs. Browse by frequently
searched topics or find the academic support office you are looking for at
<a href="https://www.mines.edu/undergraduate-studies/tutoring/">mines.edu/undergraduate-studies/tutoring</a>.</p>

<a id="p00-15"></a>
<h4>15. Expectations for Participation <span style="font-size:0.82em">&nbsp;[<a href="#toc-010">back to list &uarr;</a>]</span></h4>
<p>You are expected to engage in all course activities, tasks, and assignments as an emerging professional. You are expected to spend
additional time on this course each week outside class, reading and reproducing the worked examples in the notes before class, working on labs
and quizzes, in addition to actively participating during the lecture and lab times each week.</p>